# 00 · Workshop Environment

## The story we follow all day
Tenants email DLD every day asking whether their landlord's rent increase is allowed. The answer depends on Dubai's rules: how far the rent is below the area average, and whether the landlord gave 90 days' notice. Sara, a DLD rental-services officer, gets an AI team to read the email, check the rules, do the maths, draft the reply and book a mediation appointment, and nothing is sent or booked until she approves.

```
                Tenant email in Sara's Outlook inbox
                              │
         ┌────────────────────▼────────────────────┐
         │  Manager agent: plans and coordinates   │
         └──────┬───────────────┬──────────────┬───┘
                ▼               ▼              ▼
           Mail agent      Rules agent    Calendar agent
         reads & replies  Foundry IQ +    books mediation
          (Work IQ Mail)  citations       (Work IQ Calendar)
```

| # | Notebook | What it shows |
|---|---|---|
| 00 | Workshop Environment | Check prerequisites and build your own private copy in Azure |
| 01 | Knowledge, RAG and Foundry IQ | No grounding vs. classic search vs. Foundry IQ, and five ways to stop made-up answers |
| 02 | Safety, Identity and Evaluation | The Rules agent with its own identity, attack detection, and automated grading |
| 03 | Models and Model Routing | Letting Model Router pick a cheaper or stronger model per question |
| 04 | Agent Tools: MCP and A2A | Mail and Calendar agents acting as you, with approval before every write; agent-to-agent |
| 05 | Agent Framework and Hosted Agents | Shipping the coded manager agent as a container with its own identity |
| 06 | Multi-Agent Orchestration and Events | The team in three modes: you in charge, you give a goal, an email starts it |
| 07 | Observability, Governance and Delivery | Traces, cost, publishing, and **cleanup** |

## Prerequisites

Go through this list **before the session**. Step 4 checks most of it automatically.

| Need | Why | How to check or get it |
|---|---|---|
| A Google account | The notebooks run in Google Colab (the free tier is enough: the heavy work runs in Azure) | Sign in at colab.research.google.com |
| An Azure subscription you can bill to | Everything is created in **your** subscription | portal.azure.com → Subscriptions |
| **Owner** on the subscription, or on a resource group: **Contributor + Role Based Access Control Administrator** | The template creates resources **and** gives each agent narrow access through its own identity (no keys) | Checked automatically in step 4 |
| Model quota in **Sweden Central**: 100K tokens/min for `gpt-5.4-mini` and `model-router`, 50K for `text-embedding-3-large` | The three model deployments | Checked automatically in step 4 |
| A free slot for Azure AI Search, **or** use `basic` | Each subscription can have only one Free Search service | Checked automatically in step 4 |
| A work account with Outlook, and a **Microsoft 365 Copilot** licence | The Mail and Calendar agents use the Work IQ tools, acting as you | Checked in step 4 (warning only: without it, watch the facilitator for those steps) |
| Permission to sign in to Microsoft Foundry (ai.azure.com) in the same tenant | Notebook 04 has one portal step: connecting the Work IQ tools | Open ai.azure.com |

### Cost (approximate, until you clean up)
- **Azure AI Search:** Free costs nothing; Basic is about **US$2.50 per day**.
- **Container Registry (Basic, from notebook 05):** about **US$0.17 per day**.
- **Model calls:** pay per use; the whole workshop is typically a few US dollars.
- Storage, monitoring and the Logic App: cents at this volume.

Nothing expires on its own. **Run the cleanup cell at the end of notebook 07** to stop all charges.

## Step 1 · Get the workshop code

In [ ]:
# @title Get the workshop code (a few seconds)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()
print("✅ Workshop code ready")

## Step 2 · Sign in to Azure

You'll see a link and a short code. Open the link, enter the code, and sign in with your **work** account. Each notebook asks for this once.

In [ ]:
print("Signed in as:", WorkshopEnvironment.signed_in_identity(credential)["user_name"])

## Step 3 · Your settings, worked out for you

Nothing to type. Your subscription is detected, a unique name is made from your sign-in, and the Search tier switches to Basic if your subscription's single Free slot is already taken. If you have **several** subscriptions, the cell tells you to paste one ID.

In [ ]:
SUBSCRIPTION_ID = ""   # only needed if you have more than one subscription

settings = WorkshopEnvironment.automatic_settings(credential, SUBSCRIPTION_ID or None)
table([
    {"setting": "Subscription", "value": settings.subscription_id},
    {"setting": "Resource group", "value": settings.resource_group},
    {"setting": "Unique suffix", "value": settings.suffix},
    {"setting": "Foundry and models", "value": settings.location},
    {"setting": "Azure AI Search", "value": f"{settings.search_sku} in {settings.search_location}"},
])

## Step 4 · Automatic prerequisite check

Every row should be ✅. A ⚠️ is a warning (for example no Copilot licence: you can still do everything except the Mail and Calendar steps). A ❌ must be fixed first; the detail column says how.

In [ ]:
from dld_workshop.preflight import run_preflight, assert_ready

results = run_preflight(credential, settings)
checklist(results)
assert_ready(results)
print("✅ Ready to deploy")

## Step 5 · Build your environment

Creates a Foundry account and project with three model deployments, Azure AI Search, Storage, monitoring, a content-safety guardrail, and the Logic App used in notebook 06, all connected together.

**Takes about 5 to 12 minutes** (most of it is Azure AI Search). Running it again is safe: it updates the same resources.

In [ ]:
environment = WorkshopEnvironment(credential, settings)
settings = environment.provision(template_path=ROOT / "infra" / "main.json")

table([
    {"resource": "Foundry project", "value": settings.project_name},
    {"resource": "Azure AI Search", "value": f"{settings.search_name} ({settings.search_sku}, {settings.search_location})"},
    {"resource": "Storage", "value": settings.storage_name},
    {"resource": "Content-safety guardrail", "value": (settings.rai_policy_id or "").rsplit("/", 1)[-1]},
    {"resource": "Email trigger (Logic App)", "value": settings.logic_app_name},
])

## Next: open notebook 01
Open the next notebook from the workshop site, or from the links in the repository README. Every notebook finds this environment by itself from your sign-in.

> ### 👉 Follow along in the portal
>
> The notebook and the portals show the **same** resources; anything you create in one appears in the other.
> 1. **Azure portal** → Resource groups → your group: Foundry, Search, Storage, Log Analytics, Application Insights, a Logic App and an Outlook connection.
> 2. **Foundry portal** (ai.azure.com) → your project → **Models**: `gpt-5.4-mini`, `model-router`, `text-embedding-3-large`.
> 3. **Foundry portal** → **Guardrails**: `dld-workshop-strict`.

---
## Words you'll see
| Word | Plain meaning |
|---|---|
| **Agent** | An AI assistant with instructions, tools and its own identity |
| **RAG** | Letting the AI look things up in your documents before answering |
| **Foundry IQ** | Foundry's managed "look it up" service: it plans searches across your documents and cites sources |
| **MCP** | A standard plug that connects an agent to a tool (here: Outlook mail and calendar, and the knowledge base) |
| **A2A** | A standard way for one agent to hand a task to another agent |
| **Work IQ** | Microsoft's tools that let agents work with your Microsoft 365 data, as you |
| **RERA Rental Index** | The official average rent for similar units, used to decide the allowed increase |